In [1]:
import os

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# run script when image is run
CMD ["python3", "script.py"]

Writing Dockerfile


### Write ```requirements.txt``` to local drive

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

boto3==1.24.59
numpy==1.23.4
pandas==1.2.4
catboost==1.0.4
scikit_learn==0.24.1

Writing requirements.txt


### Write ```script.py``` to local drive

In [4]:
%%writefile script.py

import os
import pandas as pd
import numpy as np
import catboost as cb
import sklearn.metrics as skm
import boto3
import pickle

# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # init client
    cls_client = boto3.client(
        's3',
    )
    # download file
    cls_client.download_file(
        str_project, 
        str_bucket_path, 
        str_local_path,
    )

# upload to s3
def upload_to_s3(str_local_path, str_bucket_path, str_project):
    boto3.resource('s3').Bucket(str_project).Object(str_bucket_path).upload_file(str_local_path)

# constants
str_project = '20231010-gen-xii'
str_task = '14_monitoring'
str_subtask = '06_tuning'
str_indicator = '30_90'
str_model = 'noPTImodel10'

str_target = f'Early_Pay_Delinquency_{str_indicator}_Flag'

int_n_tuning_jobs = 100
str_eval_metric = 'Logloss'
int_n_iterations = 1000
flt_prop_early_stopping = 0.05
dict_monotone_constraints = {
    # better
    'fltgrossmonthly__income_sum': -1, # as income increases, prediction gets better
    'fltapproveddowntotal__app': -1,
    'fltdowncash__app': -1,
    'bookvalue__app': -1,
    'ENG-dealership_age': -1,
    # worse
    'fltgrossmonthly__income_count': 1, # as count of income increases, prediction gets worse
    'ENG-loan_to_value': 1,
    'ENG-payment_to_income': 1,
    'ENG-vehicle_age': 1,
    'fltadvance__app': 1,
    'bigmileage_odometer__app': 1,
    'amtfinanced__app': 1,
    'miles_odometer__app': 1,
    'pti__app': 1,
}

# get features in model
print('Getting features in model...')
str_filename = 'final_model.pkl'
str_bucket_path = f'02_pricing_pd/02_model/{str_model}/03_final_model/{str_filename}'
str_local_path = f'./{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)
# import
cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
# rm
os.remove(str_local_path)
# get features in model
list_cols_model = list(cls_model_inference.feature_names_)

# subset dict
print('Subsetting monotone constraints...')
dict_monotone_constraints = {key: val for key, val in dict_monotone_constraints.items() if key in list_cols_model}

# get the targets
print('Getting early indicator targets...')
# get the early indicator targets
list_cols = ['bigAccountId', str_target]
str_uri = f's3://20240327-genxii-v2/02_target_creation/01_classification/df_targets.gzip'
df_tmp = pd.read_parquet(str_uri, columns=list_cols)

### PREP TRAINING DATA ###

# get the train bigaccount id
str_uri = f's3://{str_project}/02_pricing_pd/01_data_prep/03_train_valid_test_split/df_train_raw.gzip'
list_big_account_id = list(pd.read_parquet(str_uri, columns=['bigaccountid__app'])['bigaccountid__app'])

# read training data
print('Reading training data...')
str_uri = f's3://{str_project}/02_pricing_pd/02_model/{str_model}/00_preprocessing/02_make_dfs/df_train_noleaks_pre.gzip'
df_train = pd.read_parquet(
    str_uri,
    columns=list_cols_model,
)
# assign
df_train['bigAccountId'] = list_big_account_id

# join
print('Joining early indicator target...')
df_train = pd.merge(
    left=df_train,
    right=df_tmp,
    on='bigAccountId',
    how='left',
)
# rename
dict_rename = {
    str_target: 'target',
}
df_train.rename(columns=dict_rename, inplace=True)

# get the non numeric feats
print('Getting list of non-numeric columns...')
list_cols_non_numeric = []
for col in list_cols_model:
    if df_train[col].dtype not in ['float64','int64']:
        list_cols_non_numeric.append(col)

# pool training data
print('Pooling training data...')
# pool data
pool_train = cb.Pool(
    df_train[list_cols_model], 
    df_train['target'], 
    cat_features=list_cols_non_numeric,
)

### PREP VALIDATION DATA ###

# get the valid bigaccount id
str_uri = f's3://{str_project}/02_pricing_pd/01_data_prep/03_train_valid_test_split/df_valid_raw.gzip'
list_big_account_id = list(pd.read_parquet(str_uri, columns=['bigaccountid__app'])['bigaccountid__app'])

# read validation data
print('Reading validation data...')
str_uri = f's3://{str_project}/02_pricing_pd/02_model/{str_model}/00_preprocessing/02_make_dfs/df_valid_noleaks_pre.gzip'
df_valid = pd.read_parquet(
    str_uri,
    columns=list_cols_model,
)
# assign
df_valid['bigAccountId'] = list_big_account_id

# join
print('Joining early indicator target...')
df_valid = pd.merge(
    left=df_valid,
    right=df_tmp,
    on='bigAccountId',
    how='left',
)
# rename
dict_rename = {
    str_target: 'target',
}
df_valid.rename(columns=dict_rename, inplace=True)

# pool validation data
print('Pooling validation data...')
# pool data
pool_valid = cb.Pool(
    df_valid[list_cols_model], 
    df_valid['target'], 
    cat_features=list_cols_non_numeric,
)

### FIT MODEL ###

# class weights
print('Getting class weights...')
# these are coming from the test data set in gen 12 v2
flt_desired_0_prop = 0.8726371242640224
flt_desired_1_prop = 0.1273628757359776
# get the current proportions
ser_prop = df_train['target'].value_counts(normalize=True)
# get currnt percentages
flt_current_0_prop = ser_prop[0]
flt_current_1_prop = ser_prop[1]
# get the weights
flt_0_weight = flt_desired_0_prop / flt_current_0_prop
flt_1_weight = flt_desired_1_prop / flt_current_1_prop
list_class_weights = [flt_0_weight, flt_1_weight]

# iterate through LRs
list_flt_learning_rate = list(np.around(np.linspace(0.001, 0.999, int_n_tuning_jobs), 4))
dict_models = {}
list_dict_row = []
for flt_learning_rate in list_flt_learning_rate:
    # init class
    cls_model_inference = cb.CatBoostClassifier(
        task_type='CPU',
        nan_mode='Min',
        random_state=42,
        eval_metric=str_eval_metric,
        iterations=int_n_iterations,
        learning_rate=flt_learning_rate,
        class_weights=list_class_weights,
        monotone_constraints=dict_monotone_constraints,
    )
    # fit
    cls_model_inference.fit(
        pool_train,
        eval_set=[pool_valid],
        verbose=100,
        use_best_model=True,
        early_stopping_rounds=int(round(int_n_iterations*flt_prop_early_stopping)), 
    )
    
    # get training eval metric
    df_train['y_hat'] = cls_model_inference.predict_proba(df_train[cls_model_inference.feature_names_])[:,1]
    flt_mean = df_train['y_hat'].mean()
    print(f'Mean of training predictions: {flt_mean:0.4f}')
    flt_eval_metric_train = skm.log_loss(y_true=df_train['target'], y_pred=df_train['y_hat'])
    
    # get validation eval metric
    df_valid['y_hat'] = cls_model_inference.predict_proba(df_valid[cls_model_inference.feature_names_])[:,1]
    flt_eval_metric_valid = skm.log_loss(y_true=df_valid['target'], y_pred=df_valid['y_hat'])
    flt_mean = df_valid['y_hat'].mean()
    print(f'Mean of validation predictions: {flt_mean:0.4f}')
    
    # create row
    dict_row = {
        'learning_rate': flt_learning_rate,
        'train_score': flt_eval_metric_train,
        'valid_score': flt_eval_metric_valid,
    }
    list_dict_row.append(dict_row)
    
    # make df
    df_tmp = pd.DataFrame(list_dict_row)
    
    # sort
    df_tmp.sort_values(by='valid_score', ascending=True, inplace=True)
    
    # save
    str_filename = 'df_tuning.csv'
    str_uri = f's3://{str_project}/{str_task}/{str_subtask}/{str_indicator}/{str_filename}'
    df_tmp.to_csv(str_uri, index=False)
    
    # assign
    dict_models[flt_learning_rate] = cls_model_inference

# find the model with the best lr
df_tmp.sort_values(by='valid_score', ascending=True, inplace=True)

# get the lr
flt_learning_rate = df_tmp['learning_rate'].iloc[0]

# get that model
cls_model_inference = dict_models[flt_learning_rate]

# save locally
str_filename = 'cls_model_inference.pkl'
str_local_path = f'./{str_filename}'
pickle.dump(cls_model_inference, open(str_local_path, 'wb'))

# upload
str_bucket_path = f'{str_task}/{str_subtask}/{str_indicator}/{str_filename}'
upload_to_s3(
    str_local_path=str_local_path,
    str_bucket_path=str_bucket_path,
    str_project=str_project,
)

Writing script.py


### Build and push to ECR

In [5]:
%%sh

# name the image
image=genxii-early-30-90

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 362B done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.9
#2 DONE 0.3s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [1/6] FROM docker.io/library/python:3.9@sha256:c17c71e1f5f258803a6b7c391f8013adbf84285af54c2a811de4a5a1ac5a8676
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 8.23kB done
#5 DONE 0.0s

#6 [2/6] RUN apt-get update
#6 CACHED

#7 [4/6] COPY requirements.txt .
#7 CACHED

#8 [3/6] RUN pip install --upgrade pip
#8 CACHED

#9 [5/6] RUN pip install -r requirements.txt
#9 CACHED

#10 [6/6] COPY script.py .
#10 DONE 0.0s

#11 exporting to image
#11 exporting layers 0.0s done
#11 writing image sha256:0e3f3eca780d60856d8a56a7b622c7fe0b10a64c4ba7f6c9a598b4a03424c5bb done
#11 naming to docker.io/library/genxii-early-30-90 done
#11 DONE 0.0s
WARNING! Your passwor

Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-early-30-90' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-early-30-90]
698f3f12d4a3: Preparing
02506c5efead: Preparing
6cce1717a9d0: Preparing
87b0dc1f67ea: Preparing
57d96ce2aac8: Preparing
60a159600b22: Preparing
ee959616fc20: Preparing
d0e85779261a: Preparing
dafb8aed9f7f: Preparing
41d4dc7516bb: Preparing
c0f51bbdc37d: Preparing
91b542912d12: Preparing
60a159600b22: Waiting
91b542912d12: Waiting
ee959616fc20: Waiting
41d4dc7516bb: Waiting
c0f51bbdc37d: Waiting
dafb8aed9f7f: Waiting
57d96ce2aac8: Layer already exists
6cce1717a9d0: Layer already exists
87b0dc1f67ea: Layer already exists
02506c5efead: Layer already exists
ee959616fc20: Layer already exists
60a159600b22: Layer already exists
dafb8aed9f7f: Layer already exists
91b542912d12: Layer already exists
c0f51bbdc37d: Layer already exists
d0e85779261a: Layer already exists
41d4dc7516bb: Layer already exists
698f3f12d4a3: Pushed
latest: digest: sha256:112bdddc6b57f6e274f1da41b880f416f666235ea7a85504babab80

### Clean-up

In [6]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass